---
# 🤖 Azure ML — Pipeline completo

## **Fase 1: Configuración inicial**

### 1.1 Conexión con Azure Machine Learning

Configuramos el cliente de Azure ML y las importaciones necesarias.

> ⚠️ **Importante:** Esta celda debe ejecutarse **siempre primero**. Sin ella, el resto de celdas fallan porque necesitan `mlclient`.

In [6]:
import os
import mlflow
from azure.ai.ml import MLClient
from azure.ai.ml import automl
from azure.ai.ml import Input
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

print("⏳ Cargando variables de entorno desde el archivo .env...")
load_dotenv()

if not os.getenv("AZURE_WORKSPACE_NAME"):
    raise ValueError("❌ Error: No se encontraron las variables en el archivo .env. Revisa el archivo o la ruta.")

print(f"✅ Variables cargadas. Conectando al Workspace: '{os.getenv('AZURE_WORKSPACE_NAME')}'...")

# Crear el cliente de Azure ML (se usará en todas las celdas siguientes)
mlclient = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)

# ⚠️ NO usar mlflow.set_tracking_uri() cuando se usa azureml-mlflow
# El plugin azureml-mlflow ya configura la conexión automáticamente.
# Si se asigna manualmente, da conflicto con azureml_artifacts_builder.

print(f"\n✅ Cliente Azure ML creado correctamente: {mlclient.workspace_name}")
print("💡 El tracking de MLflow se configura automáticamente vía azureml-mlflow")

⏳ Cargando variables de entorno desde el archivo .env...
✅ Variables cargadas. Conectando al Workspace: 'ml-project-demo'...


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented



✅ Cliente Azure ML creado correctamente: ml-project-demo
💡 El tracking de MLflow se configura automáticamente vía azureml-mlflow


---
### 1.2 Estructura del dataset (MLTable)

Estructura de archivos que Azure ML necesita para interpretar los datos:

```
data/
└── test-data/
    ├── diabetes-test.csv    ← Datos de diabetes
    └── MLTable              ← YAML de configuración para Azure ML
```

In [7]:
# Registrar el dataset como recurso MLTable en Azure
# Si ya existe, lo reutiliza; si no, lo crea
try:
    dataset = mlclient.data.get(
        name="diabetes-dataset", 
        version="1"
        )
    print(f"♻️ Dataset 'diabetes-dataset:1' ya existe. Reutilizando...")
except:
    dataset = Data(
        name="diabetes-dataset",
        version="1",
        description="Dataset de diabetes para AutoML",
        path="./data/test-data",
        type=AssetTypes.MLTABLE
    )
    mlclient.data.create_or_update(dataset)
    print("✅ Dataset 'diabetes-dataset:1' registrado en Azure ML")

♻️ Dataset 'diabetes-dataset:1' ya existe. Reutilizando...


---
## **Fase 2: Registro de datos en Azure ML**

### 2.1 Registrar el dataset como recurso MLTable

Subimos el dataset local a la nube para que AutoML pueda acceder a él.

In [13]:
# Crear la referencia de entrada al dataset registrado en Azure
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:diabetes-dataset:1"
)

# Configurar la tarea de clasificación
# auro vamos a crear un experimento de AutoML para clasificación
# automl buscara el mejor modelo de clasificación para predecir la columna "target" del dataset
job = automl.classification(
    experiment_name="diabetes-automl",
    training_data=my_training_data_input,  # ← Usamos el Input, NO el objeto Data
    target_column_name="target",
    compute="cpu-cluster",
    primary_metric="accuracy"
    # featurization="auto",  
    # "auto" los datos se preprocesan (normalizan, escala, imputan) automáticamente (por defecto)
    # "off" ← los datos no se modifican
    # "custom" ← se puede definir un diccionario de preprocesamiento personalizado (ver documentación AzureML)
)

job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

# Lanzar el experimento
returned_job = mlclient.jobs.create_or_update(job)
print(f"🚀 Job AutoML iniciado: {returned_job.name}")
print(f"📌 Sigue el progreso en Azure ML Studio → Jobs → {returned_job.name}")

🚀 Job AutoML iniciado: musing_diamond_185w7nkjbn
📌 Sigue el progreso en Azure ML Studio → Jobs → musing_diamond_185w7nkjbn


---
## **Fase 3: AutoML (clasificación)**

### 3.1 Configurar y ejecutar AutoML

Creamos un **Input** que apunta al dataset ya registrado en la nube y lanzamos el experimento de clasificación.

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `experiment_name` | `diabetes-automl` | Nombre del experimento en Azure ML |
| `target_column_name` | `Diabetic` | Columna objetivo (0 = No diabético, 1 = Diabético) |
| `compute` | `cpu-cluster` | Clúster de cómputo |
| `primary_metric` | `accuracy` | Métrica principal para optimizar |
| `max_trials` | 5 | Número máximo de algoritmos a probar |
| `timeout_minutes` | 60 | Tiempo máximo total del experimento |

In [9]:
# Verificar dataset registrado
registered_data = mlclient.data.get(name="diabetes-dataset", version="1")
print(f"📦 Dataset: {registered_data.name} v{registered_data.version}")

# Mostrar estado del job
print(f"📊 Estado del job: {returned_job.status}")
print("🔗 Ve a https://ml.azure.com → Experiments → diabetes-automl para ver el progreso")

📦 Dataset: diabetes-dataset v1
📊 Estado del job: NotStarted
🔗 Ve a https://ml.azure.com → Experiments → diabetes-automl para ver el progreso


---
## **Fase 5: Entrenamiento por comandos (YAML)**

### 5.1 Ejecutar job desde `job.yml`

Lanzamos el entrenamiento directamente desde el archivo YAML configurado con `LogisticRegression`.

In [10]:
from azure.ai.ml import load_job
from azure.ai.ml.entities import Job

# Cargar job desde el archivo YAML
job_from_yml = load_job(source="./src/job.yml")

# Lanzar el job
returned_job_yml = mlclient.jobs.create_or_update(job_from_yml)
print(f"🚀 Job desde YAML iniciado: {returned_job_yml.name}")
print(f"📌 Ver en Azure ML Studio → Jobs → {returned_job_yml.name}")

ResourceNotFoundError: (UserError) No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 
Code: UserError
Message: No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 

---
## **Fase 4: Verificación**

### 4.1 Comprobar dataset y estado del job AutoML

---
## **Fase 6: MLflow — Seguimiento de experimentos**

### 6.1 Creación de un experimento con MLflow

Basado en la [guía oficial de MLflow](https://mlflow.org/docs/latest/ml/tracking/#automatic-logging?azure-portal=true).

---

#### 📌 `mlflow.start_run()`

Crea una ejecución de MLflow para registrar métricas y parámetros:

```python
with mlflow.start_run():
    mlflow.log_param("lr", 0.001)
    # Código de entrenamiento
    mlflow.log_metric("val_loss", val_loss)
```

---

#### 🔄 `mlflow.autolog()` — Registro automático

Habilita el seguimiento automático de métricas para frameworks compatibles: scikit-learn, TensorFlow, PyTorch, **XGBoost**, LightGBM y Keras.

---

#### ✍️ `mlflow.xgboost.autolog()` — Registro personalizado

Para registrar manualmente (sin autolog), puedes usar `mlflow.log_param()` y `mlflow.log_metric()` de forma explícita.

In [ ]:
# Establecer experimento de MLflow
mlflow.set_experiment(experiment_name="diabetes-automl")
# Si no se especifica, MLflow usaría el nombre "default"

<Experiment: artifact_location='file:///home/renato/Desktop/MLOps-practicas/mlruns/296912912954074563', creation_time=1783680076689, experiment_id='296912912954074563', last_update_time=1783680076689, lifecycle_stage='active', name='diabetes-automl', tags={}>

### 6.5 Cargar datos para ejemplos de entrenamiento

Cargamos el CSV local y dividimos en train/test para los ejemplos con XGBoost.

In [ ]:
# Cargar datos locales y dividir en train/test
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("./data/test-data/diabetes-test.csv")
print(f"📊 Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

X = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
        'SerumInsulin','BMI','DiabetesPedigree','Age']].values
y = df['Diabetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
print(f"✅ Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

📊 Datos cargados: 5000 filas, 9 columnas
✅ Train: 3500 samples | Test: 1500 samples


In [ ]:
# 6.3 Ejemplo: Registro automático con XGBoost
from xgboost import XGBClassifier

with mlflow.start_run():
    mlflow.autolog()

    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

2026/07/10 12:42:40 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.9.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


2026/07/10 12:42:40 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/07/10 12:42:40 WARNING mlflow.utils.autologging_utils: MLflow xgboost autologging is known to be compatible with 1.4.2 <= xgboost <= 3.0.0, but the installed version is 3.3.0. If you encounter errors during autologging, try upgrading / downgrading xgboost to a compatible version, or try upgrading MLflow.
2026/07/10 12:42:40 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/07/10 12:42:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/xgboost/sklearn.py:1122: UserWarning: [12:42:41] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1564: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats."


In [ ]:
# 6.4 Ejemplo: Registro manual con XGBoost
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

with mlflow.start_run():
    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

NameError: name 'mlflow' is not defined

---
## **Fase 7: IA Responsable (RAI Dashboard)**

### 7.1 Entrenar modelo base para RAI

Entrenamos un `RandomForestClassifier` que servirá como base para generar el dashboard de IA Responsable.

Evaluamos el modelo con las herramientas de Responsible AI: equidad, explicabilidad, contrafactuales y análisis causal.

> 📎 [Documentación oficial de RAI Dashboard](https://learn.microsoft.com/es-es/training/modules/experiment-azure-machine-learning/7-responsible-ai-dashboard?tabs=error-analysis)

In [ ]:
# Entrenar RandomForest para el panel RAI
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

modelo_rai = RandomForestClassifier(n_estimators=100, random_state=0)
modelo_rai.fit(X_train, y_train)

y_pred_rai = modelo_rai.predict(X_test)
y_proba_rai = modelo_rai.predict_proba(X_test)

print("📊 Reporte de clasificación del modelo para RAI:")
print(classification_report(y_test, y_pred_rai, target_names=["No Diabético", "Diabético"]))

2026/07/10 12:42:56 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd08c1112d0324477807cf4f635cd0fd3', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


📊 Reporte de clasificación del modelo para RAI:
              precision    recall  f1-score   support

No Diabético       0.92      0.95      0.94       980
   Diabético       0.90      0.85      0.88       520

    accuracy                           0.92      1500
   macro avg       0.91      0.90      0.91      1500
weighted avg       0.92      0.92      0.92      1500



### 7.2 Registrar modelo y crear dashboard RAI

Guardamos el modelo, lo registramos en Azure ML y generamos las instrucciones para crear el panel RAI desde Studio.

In [ ]:
import joblib

# 1. Guardar el modelo entrenado localmente
os.makedirs("rai_model", exist_ok=True)
joblib.dump(modelo_rai, "rai_model/model.pkl")
print("✅ Modelo guardado en rai_model/model.pkl")

# 2. Registrar el modelo en Azure ML
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_rai = Model(
    name="diabetes-rai-model",
    version="2",
    path="./rai_model",
    type=AssetTypes.CUSTOM_MODEL,
    description="RandomForest para diabetes - RAI dashboard"
)
mlclient.models.create_or_update(model_rai)
print("✅ Modelo 'diabetes-rai-model:2' registrado en Azure ML")

# 3. Crear pipeline RAI con componentes del registry
# NOTA: Para ejecutar este pipeline necesitas los componentes RAI
# en tu workspace. Si no los tienes, puedes instalarlos via:
# az ml component create --file rai_component.yml
# O usar el dashboard desde Azure ML Studio directamente.

print("\n" + "="*60)
print("📋 PARA VER EL PANEL RAI DESDE AZURE ML STUDIO:")
print("="*60)
print("1. Ve a https://ml.azure.com")
print("2. En el menú izquierdo → Models → diabetes-rai-model")
print("3. Ve a la pestaña 'Responsible AI'")
print("4. Haz clic en 'Create Responsible AI dashboard'")
print("5. Selecciona el cluster 'cpu-cluster' y la columna objetivo 'Diabetic'")
print("6. Elige las herramientas que quieras incluir:")
print("   - Error Analysis (análisis de errores)")
print("   - Model Explanations (explicabilidad - Feature Importance)")
print("   - Counterfactuals (contrafactuales)")
print("   - Causal Analysis (análisis causal)")
print("="*60)

✅ Modelo guardado en rai_model/model.pkl


Uploading rai_model (4.33 MBs): 100%|██████████| 4326493/4326493 [00:02<00:00, 1491583.23it/s]




✅ Modelo 'diabetes-rai-model:2' registrado en Azure ML

📋 PARA VER EL PANEL RAI DESDE AZURE ML STUDIO:
1. Ve a https://ml.azure.com
2. En el menú izquierdo → Models → diabetes-rai-model
3. Ve a la pestaña 'Responsible AI'
4. Haz clic en 'Create Responsible AI dashboard'
5. Selecciona el cluster 'cpu-cluster' y la columna objetivo 'Diabetic'
6. Elige las herramientas que quieras incluir:
   - Error Analysis (análisis de errores)
   - Model Explanations (explicabilidad - Feature Importance)
   - Counterfactuals (contrafactuales)
   - Causal Analysis (análisis causal)


### 7.3 Componentes del panel RAI

| Componente | Función |
|------------|---------|
| **Error Analysis** | Identifica segmentos con mayor tasa de error |
| **Model Explanations** | Muestra importancia de cada característica en las predicciones |
| **Counterfactuals** | Simula cambios en features para alterar la predicción |
| **Causal Analysis** | Descubre relaciones causa-efecto entre features y target |
| **Fairness** | Evalúa sesgos del modelo en distintos grupos demográficos |

---
## **Fase 8: Resultados**

### 8.1 Consultar el mejor algoritmo encontrado por AutoML

Una vez que el job AutoML haya finalizado, consultamos cuál fue el mejor modelo y sus métricas.

In [12]:
# Consultar estado y resultados del AutoML
job_status = mlclient.jobs.get(name=returned_job.name)
print(f"📊 Estado del job AutoML: {job_status.status}")

if job_status.status == "Completed":
    print("\n🏆 AutoML ha terminado. El mejor modelo fue:")
    print(f"   Algoritmo: {job_status.trial.component.name if hasattr(job_status.trial, 'component') else 'Ver en Azure ML Studio'}")
    
    # Obtener las métricas del mejor modelo
    best_run = list(mlclient.jobs.list(parent_job_name=returned_job.name))[0]
    metrics = mlclient.jobs.get(name=best_run.name if hasattr(best_run, 'name') else returned_job.name)
    print(f"   Accuracy: {metrics.metrics.get('accuracy', 'N/A')}")
    
elif job_status.status == "Running":
    print("\n⏳ AutoML sigue ejecutándose...")
    print("ℹ️  Puedes ver el progreso en:")
    print(f"   https://ml.azure.com/experiments/diabetes-automl/jobs/{returned_job.name}")
else:
    print(f"\n⚠️ Estado inesperado: {job_status.status}")
    print("ℹ️  Revisa en Azure ML Studio para más detalles")

📊 Estado del job AutoML: Running

⏳ AutoML sigue ejecutándose...
ℹ️  Puedes ver el progreso en:
   https://ml.azure.com/experiments/diabetes-automl/jobs/lemon_machine_lhv36wvbcc
